In [1]:
# Periksa konfigurasi Training SSD

from pathlib import Path
import re
import sys

BASE = Path("/home/rzangef/SSD-PPE")
TRAIN_DIR = BASE / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
PIPELINE = (
    BASE / "models/ssd_mobilenet_v2_fpnlite_320x320/"
    "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config"
)
INFER_PIPELINE = (
    BASE / "inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config"
)
DATASET = Path("/home/rzangef/spark-data/CHVG-SSD")

print("Python:", sys.executable)
print("Checkpoint folder:", TRAIN_DIR.exists())
print("Checkpoint config:", (TRAIN_DIR / "checkpoint").exists())
print("Training pipeline:", PIPELINE.exists())
print("Inference pipeline:", INFER_PIPELINE.exists())
print("Dataset folder:", DATASET.exists())

print("\n=== CHECKPOINT FILES ===")
for name in ["ckpt-24.index", "ckpt-24.data-00000-of-00001"]:
    p = TRAIN_DIR / name
    print(name, "exists =", p.exists(),
          "| size =", f"{p.stat().st_size / 1024**2:.2f} MB" if p.exists() else "-")

print("\n=== KONFIGURASI TRAINING ===")
if PIPELINE.exists():
    config = PIPELINE.read_text()
    for key in ["batch_size", "num_steps", "num_classes", "input_path",
                "label_map_path", "fine_tune_checkpoint"]:
        values = re.findall(
            rf"(?m)^\s*{key}\s*:\s*['\"]?([^'\"\n]+)",
            config
        )
        print(f"{key}: {values[:10]}")


Python: /home/rzangef/SSD-PPE/.venv/bin/python
Checkpoint folder: True
Checkpoint config: True
Training pipeline: True
Inference pipeline: True
Dataset folder: True

=== CHECKPOINT FILES ===
ckpt-24.index exists = True | size = 0.05 MB
ckpt-24.data-00000-of-00001 exists = True | size = 19.85 MB

=== KONFIGURASI TRAINING ===
batch_size: ['128']
num_steps: ['50000']
num_classes: ['90']
input_path: ['PATH_TO_BE_CONFIGURED', 'PATH_TO_BE_CONFIGURED']
label_map_path: ['PATH_TO_BE_CONFIGURED', 'PATH_TO_BE_CONFIGURED']
fine_tune_checkpoint: ['PATH_TO_BE_CONFIGURED']


In [2]:

from pathlib import Path
import re

BASE = Path("/home/rzangef/SSD-PPE")
DATASET = Path("/home/rzangef/spark-data/CHVG-SSD")
TRAIN_DIR = BASE / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"

# 1. Periksa informasi checkpoint
print("=" * 60)
print("1. ISI FILE CHECKPOINT")
print("=" * 60)

checkpoint_state = TRAIN_DIR / "checkpoint"
if checkpoint_state.exists():
    print(checkpoint_state.read_text())
else:
    print("File checkpoint tidak ditemukan.")

# 2. Cari semua pipeline.config di proyek SSD
print("\n" + "=" * 60)
print("2. SEMUA FILE pipeline.config")
print("=" * 60)

for p in BASE.rglob("pipeline.config"):
    print(p)

# 3. Cari file dataset, label map, dan konfigurasi
print("\n" + "=" * 60)
print("3. FILE DATASET DAN LABEL MAP")
print("=" * 60)

for root in [DATASET, BASE]:
    if root.exists():
        for p in root.rglob("*"):
            if not p.is_file():
                continue

            name = p.name.lower()
            if (
                "label_map" in name
                or name.endswith((".record", ".tfrecord", ".tfrecords"))
                or name == "pipeline.config"
            ):
                print(p)

# 4. Periksa konfigurasi inference yang tersedia
print("\n" + "=" * 60)
print("4. PENGATURAN CONFIG INFERENCE")
print("=" * 60)

INFER = BASE / "inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config"

if INFER.exists():
    config = INFER.read_text()

    for key in [
        "batch_size",
        "num_classes",
        "num_steps",
        "input_path",
        "label_map_path",
        "fine_tune_checkpoint",
        "num_examples",
        "use_bfloat16",
    ]:
        values = re.findall(
            rf"(?m)^\s*{key}\s*:\s*['\"]?([^'\"\n]+)",
            config
        )
        print(f"{key}: {values[:10]}")
else:
    print("Config inference tidak ditemukan.")


1. ISI FILE CHECKPOINT
model_checkpoint_path: "ckpt-24"
all_model_checkpoint_paths: "ckpt-18"
all_model_checkpoint_paths: "ckpt-19"
all_model_checkpoint_paths: "ckpt-20"
all_model_checkpoint_paths: "ckpt-21"
all_model_checkpoint_paths: "ckpt-22"
all_model_checkpoint_paths: "ckpt-23"
all_model_checkpoint_paths: "ckpt-24"
all_model_checkpoint_timestamps: 1791483855.7287722
all_model_checkpoint_timestamps: 1791484142.371906
all_model_checkpoint_timestamps: 1791484431.495789
all_model_checkpoint_timestamps: 1791484718.3139415
all_model_checkpoint_timestamps: 1791485007.5112097
all_model_checkpoint_timestamps: 1791485294.1478777
all_model_checkpoint_timestamps: 1791485586.6523128
last_preserved_timestamp: 1791478895.114577


2. SEMUA FILE pipeline.config
/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config
/home/rzangef/SSD-PPE/inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config

3. FILE DATASET DAN LABEL MAP
/hom

In [3]:

from pathlib import Path
import re

BASE = Path("/home/rzangef/SSD-PPE")
TRAIN_DIR = BASE / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"

CONFIGS = {
    "Config training model": BASE / (
        "models/ssd_mobilenet_v2_fpnlite_320x320/"
        "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config"
    ),
    "Config inference": BASE / "inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config",
    "Config CHVG": Path("/home/rzangef/spark-data/CHVG-SSD/config/pipeline.config"),
}

print("=" * 65)
print("1. CHECKPOINT TRAINING")
print("=" * 65)

checkpoint_file = TRAIN_DIR / "checkpoint"
if checkpoint_file.exists():
    print(checkpoint_file.read_text())
else:
    print("File checkpoint tidak ditemukan")

print("\nFile checkpoint yang tersedia:")
for p in sorted(TRAIN_DIR.glob("ckpt-*.index")):
    print(p.name, f"| {p.stat().st_size / 1024**2:.2f} MB")

print("\n" + "=" * 65)
print("2. PEMERIKSAAN KONFIGURASI")
print("=" * 65)

keys = [
    "num_classes",
    "batch_size",
    "num_steps",
    "fine_tune_checkpoint",
    "num_examples",
    "input_path",
    "label_map_path",
    "load_all_detection_checkpoint_vars",
]

for title, path in CONFIGS.items():
    print(f"\n--- {title} ---")
    print("Path:", path)
    print("Tersedia:", path.exists())

    if not path.exists():
        continue

    content = path.read_text()
    for key in keys:
        matches = re.findall(
            rf"(?m)^\s*{re.escape(key)}\s*:\s*(.+?)\s*$",
            content
        )
        if matches:
            print(f"{key}:")
            for value in matches:
                print("  ", value)

print("\n" + "=" * 65)
print("3. FOLDER HASIL TRAINING")
print("=" * 65)

for name in ["train", "eval", "eval_ckpt24"]:
    folder = TRAIN_DIR / name
    print(f"\n{name}: tersedia = {folder.exists()}")
    if folder.exists():
        for p in sorted(folder.glob("**/events.out.tfevents.*")):
            print(" ", p)

1. CHECKPOINT TRAINING
model_checkpoint_path: "ckpt-24"
all_model_checkpoint_paths: "ckpt-18"
all_model_checkpoint_paths: "ckpt-19"
all_model_checkpoint_paths: "ckpt-20"
all_model_checkpoint_paths: "ckpt-21"
all_model_checkpoint_paths: "ckpt-22"
all_model_checkpoint_paths: "ckpt-23"
all_model_checkpoint_paths: "ckpt-24"
all_model_checkpoint_timestamps: 1791483855.7287722
all_model_checkpoint_timestamps: 1791484142.371906
all_model_checkpoint_timestamps: 1791484431.495789
all_model_checkpoint_timestamps: 1791484718.3139415
all_model_checkpoint_timestamps: 1791485007.5112097
all_model_checkpoint_timestamps: 1791485294.1478777
all_model_checkpoint_timestamps: 1791485586.6523128
last_preserved_timestamp: 1791478895.114577


File checkpoint yang tersedia:
ckpt-18.index | 0.05 MB
ckpt-19.index | 0.05 MB
ckpt-20.index | 0.05 MB
ckpt-21.index | 0.05 MB
ckpt-22.index | 0.05 MB
ckpt-23.index | 0.05 MB
ckpt-24.index | 0.05 MB

2. PEMERIKSAAN KONFIGURASI

--- Config training model ---
Path: /home/

In [4]:
# Cek log training

from pathlib import Path
import tensorflow as tf
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

BASE = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

LOG_DIRS = {
    "TRAINING": BASE / "train",
    "EVALUASI LAMA": BASE / "eval",
    "EVALUASI CKPT24": BASE / "eval_ckpt24" / "eval",
}

for title, folder in LOG_DIRS.items():
    print("\n" + "=" * 70)
    print(title)
    print("Folder:", folder)
    print("=" * 70)

    files = list(folder.glob("events.out.tfevents.*"))
    if not files:
        print("Tidak ada event log.")
        continue

    try:
        acc = EventAccumulator(
            str(folder),
            size_guidance={"scalars": 0, "tensors": 0}
        )
        acc.Reload()
        tags = acc.Tags()

        print("Scalar tags:", tags.get("scalars", []))
        print("Tensor tags:", tags.get("tensors", []))

        for tag in tags.get("scalars", []):
            events = acc.Scalars(tag)
            if events:
                last = events[-1]
                print(
                    f"{tag}: nilai terakhir={last.value:.6f}, "
                    f"step={last.step}"
                )

        for tag in tags.get("tensors", []):
            events = acc.Tensors(tag)
            if events:
                last = events[-1]
                try:
                    value = tf.make_ndarray(last.tensor_proto)
                    if value.size <= 10:
                        print(
                            f"{tag}: nilai terakhir={value.tolist()}, "
                            f"step={last.step}"
                        )
                    else:
                        print(
                            f"{tag}: tensor shape={value.shape}, "
                            f"step={last.step}"
                        )
                except Exception as err:
                    print(f"{tag}: tidak dapat dibaca ({err})")

    except Exception as err:
        print("Gagal membaca log:", repr(err))

2026-10-19 10:06:57.668509: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 10:06:58.401303: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-19 10:06:58.401796: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-19 10:06:58.546820: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-19 10:06:58.853283: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 10:06:58.856820: I tensorflow/core/platform/cpu_feature_guard.cc:1


TRAINING
Folder: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/train


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from pathlib import Path

folder = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch/train"
)

print("Folder ditemukan:", folder.exists())

for file in folder.glob("events.out.tfevents.*"):
    print("\nFile log:", file.name)

    acc = EventAccumulator(str(folder))
    acc.Reload()

    tags = acc.Tags()
    print("\nScalar tags:")
    print(tags.get("scalars", []))

    print("\nTensor tags:")
    print(tags.get("tensors", []))

    for tag in tags.get("scalars", []):
        events = acc.Scalars(tag)
        if events:
            print(
                f"{tag}: nilai terakhir={events[-1].value:.6f}, "
                f"step={events[-1].step}"
            )

Folder ditemukan: True

File log: events.out.tfevents.1791478871.LAPTOP-ODRK022O.4855.0.v2

Scalar tags:
[]

Tensor tags:
['train_input_images', 'steps_per_sec', 'Loss/localization_loss', 'Loss/classification_loss', 'Loss/regularization_loss', 'Loss/total_loss', 'learning_rate']


In [6]:

from pathlib import Path
import tensorflow as tf
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

BASE = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

folders = {
    "EVALUASI LAMA": BASE / "eval",
    "EVALUASI CHECKPOINT 24": BASE / "eval_ckpt24" / "eval",
}

for name, folder in folders.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if not folder.exists():
        print("Folder tidak ditemukan:", folder)
        continue

    acc = EventAccumulator(
        str(folder),
        size_guidance={"scalars": 0, "tensors": 0}
    )
    acc.Reload()
    tags = acc.Tags()

    print("Scalar tags:", tags.get("scalars", []))
    print("Tensor tags:", tags.get("tensors", []))

    for tag in tags.get("scalars", []):
        events = acc.Scalars(tag)
        if events:
            e = events[-1]
            print(f"{tag}: {e.value:.6f} | step={e.step}")

    for tag in tags.get("tensors", []):
        events = acc.Tensors(tag)
        if not events:
            continue

        e = events[-1]
        try:
            value = tf.make_ndarray(e.tensor_proto)
            if value.size == 1:
                print(f"{tag}: {value.item()} | step={e.step}")
            else:
                print(f"{tag}: shape={value.shape} | step={e.step}")
        except Exception as err:
            print(f"{tag}: gagal dibaca: {err}")


EVALUASI LAMA
Scalar tags: []
Tensor tags: ['eval_side_by_side_0_0', 'eval_side_by_side_1_0', 'eval_side_by_side_2_0', 'eval_side_by_side_3_0', 'eval_side_by_side_4_0', 'eval_side_by_side_5_0', 'eval_side_by_side_6_0', 'eval_side_by_side_7_0', 'eval_side_by_side_8_0', 'eval_side_by_side_9_0', 'DetectionBoxes_Precision/mAP', 'DetectionBoxes_Precision/mAP@.50IOU', 'DetectionBoxes_Precision/mAP@.75IOU', 'DetectionBoxes_Precision/mAP (small)', 'DetectionBoxes_Precision/mAP (medium)', 'DetectionBoxes_Precision/mAP (large)', 'DetectionBoxes_Recall/AR@1', 'DetectionBoxes_Recall/AR@10', 'DetectionBoxes_Recall/AR@100', 'DetectionBoxes_Recall/AR@100 (small)', 'DetectionBoxes_Recall/AR@100 (medium)', 'DetectionBoxes_Recall/AR@100 (large)', 'Loss/localization_loss', 'Loss/classification_loss', 'Loss/regularization_loss', 'Loss/total_loss']
eval_side_by_side_0_0: shape=(3,) | step=23000
eval_side_by_side_1_0: shape=(3,) | step=23000
eval_side_by_side_2_0: shape=(3,) | step=23000
eval_side_by_side_

In [7]:
# Verifikasi Checkpoint

from pathlib import Path
import tensorflow as tf

train_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

for step in range(18, 25):
    prefix = str(train_dir / f"ckpt-{step}")
    index_file = Path(prefix + ".index")

    if not index_file.exists():
        continue

    reader = tf.train.load_checkpoint(prefix)
    shape_map = reader.get_variable_to_shape_map()

    print(f"\n=== ckpt-{step} ===")

    for key in [
        "global_step",
        "global_step/.ATTRIBUTES/VARIABLE_VALUE",
    ]:
        if key in shape_map:
            print(key, "=", reader.get_tensor(key))
            break
    else:
        print("global_step tidak ditemukan dengan nama umum")

    print("Jumlah variabel:", len(shape_map))
    print("Contoh variabel:", list(shape_map.keys())[:8])


=== ckpt-18 ===
global_step tidak ditemukan dengan nama umum
Jumlah variabel: 622
Contoh variabel: ['step/.ATTRIBUTES/VARIABLE_VALUE', 'optimizer/decay/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-99/moving_variance/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-99/beta/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-97/moving_variance/.ATTRIBUTES/VARIABLE_VALUE', 'model/_feature_extractor/classification_backbone/layer_with_weights-97/moving_mean/.ATTRIBUTES/VARIABLE_VALUE']

=== ckpt-19 ===
global_step tidak ditemukan dengan nama umum
Jumlah variabel: 62

In [8]:
# Periksa Nilai Step dan Arsitektur Checkpoint

from pathlib import Path
import tensorflow as tf

train_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

for n in range(18, 25):
    prefix = str(train_dir / f"ckpt-{n}")
    if not Path(prefix + ".index").exists():
        continue

    reader = tf.train.load_checkpoint(prefix)
    variables = reader.get_variable_to_shape_map()
    step_key = "step/.ATTRIBUTES/VARIABLE_VALUE"

    step = (
        reader.get_tensor(step_key)
        if step_key in variables
        else "tidak tersedia"
    )

    print(f"ckpt-{n}: step={step}, variabel={len(variables)}")

print("\n=== PEMERIKSAAN KEPALA MODEL ===")
reader = tf.train.load_checkpoint(str(train_dir / "ckpt-24"))
variables = reader.get_variable_to_shape_map()

for key, shape in variables.items():
    if any(s in key.lower() for s in [
        "box_predictor", "class_predictor", "class_predictions"
    ]):
        print(key, shape)

ckpt-18: step=17000, variabel=622
ckpt-19: step=18000, variabel=622
ckpt-20: step=19000, variabel=622
ckpt-21: step=20000, variabel=622
ckpt-22: step=21000, variabel=622
ckpt-23: step=22000, variabel=622
ckpt-24: step=23000, variabel=622

=== PEMERIKSAAN KEPALA MODEL ===
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/3/10/moving_mean/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/4/7/beta/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_box_prediction_head/_box_encoder_layers/0/depthwise_kernel/.ATTRIBUTES/VARIABLE_VALUE [3, 3, 128, 1]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/10/moving_mean/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/4/1/beta/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/3/4/moving_variance/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_head

In [9]:

from pathlib import Path
import re

configs = {
    "Konfigurasi training model": (
        "/home/rzangef/SSD-PPE/models/"
        "ssd_mobilenet_v2_fpnlite_320x320/"
        "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/"
        "pipeline.config"
    ),
    "Konfigurasi inference": (
        "/home/rzangef/SSD-PPE/inference/"
        "ssd_mobilenet_v2_fpnlite_320x320/pipeline.config"
    ),
    "Konfigurasi CHVG": (
        "/home/rzangef/spark-data/CHVG-SSD/config/pipeline.config"
    ),
}

patterns = [
    r"^\s*model\s*\{",
    r"^\s*(ssd|ssd_mobilenet_v\d+_fpn|num_classes)\s*:",
    r"^\s*num_classes\s*:",
    r"^\s*batch_size\s*:",
    r"^\s*num_steps\s*:",
    r"^\s*fine_tune_checkpoint\s*:",
    r"^\s*input_path\s*:",
    r"^\s*label_map_path\s*:",
    r"^\s*height\s*:",
    r"^\s*width\s*:",
]

for name, filename in configs.items():
    path = Path(filename)
    print(f"\n{'=' * 65}")
    print(name)
    print("File:", filename)
    print("Ada:", path.exists())

    if not path.exists():
        continue

    lines = path.read_text(errors="replace").splitlines()

    for i, line in enumerate(lines):
        if any(re.search(pattern, line) for pattern in patterns):
            print(f"{i + 1}: {line.strip()}")


Konfigurasi training model
File: /home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config
Ada: True
1: model {
3: num_classes: 90
6: height: 320
7: width: 320
135: batch_size: 128
165: fine_tune_checkpoint: "PATH_TO_BE_CONFIGURED"
166: num_steps: 50000
175: label_map_path: "PATH_TO_BE_CONFIGURED"
177: input_path: "PATH_TO_BE_CONFIGURED"
185: label_map_path: "PATH_TO_BE_CONFIGURED"
189: input_path: "PATH_TO_BE_CONFIGURED"

Konfigurasi inference
File: /home/rzangef/SSD-PPE/inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config
Ada: True
1: model {
3: num_classes: 8
6: height: 320
7: width: 320
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps: 50000
175: label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
177: input_path: "/home/rzangef/SSD-PPE/dataset

In [10]:
# Cek file pipeline.config

from pathlib import Path
import re

root = Path("/home/rzangef/SSD-PPE")
search_roots = [
    root / "training",
    root / "inference",
    root / "models",
    Path("/home/rzangef/spark-data/CHVG-SSD"),
]

print("=== FILE PIPELINE.CONFIG ===")
for base in search_roots:
    if not base.exists():
        print("Folder tidak ditemukan:", base)
        continue

    for path in base.rglob("pipeline.config"):
        print("\nFile:", path)

        try:
            content = path.read_text(errors="replace")
        except Exception as e:
            print("Gagal membaca:", e)
            continue

        # Ringkas jenis model dan konfigurasi yang relevan
        for pattern in [
            r"^\s*model\s*\{",
            r"^\s*num_classes\s*:",
            r"^\s*height\s*:",
            r"^\s*width\s*:",
            r"^\s*batch_size\s*:",
            r"^\s*num_steps\s*:",
            r"^\s*fine_tune_checkpoint\s*:",
            r"^\s*input_path\s*:",
            r"^\s*label_map_path\s*:",
        ]:
            for line_no, line in enumerate(content.splitlines(), 1):
                if re.search(pattern, line):
                    print(f"  {line_no}: {line.strip()}")

train_dir = root / "training" / "ssd_mobilenet_v2_fpnlite_320x320_20epoch"

print("\n=== ISI FOLDER TRAINING ===")
if train_dir.exists():
    for path in sorted(train_dir.iterdir()):
        print(path.name, "| folder" if path.is_dir() else f"| {path.stat().st_size:,} bytes")
else:
    print("Folder training tidak ditemukan:", train_dir)

=== FILE PIPELINE.CONFIG ===

File: /home/rzangef/SSD-PPE/inference/ssd_mobilenet_v2_fpnlite_320x320/pipeline.config
  1: model {
  3: num_classes: 8
  6: height: 320
  7: width: 320
  135: batch_size: 1
  166: num_steps: 50000
  165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
  177: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/train.record"
  189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/val.record"
  175: label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
  185: label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"

File: /home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config
  1: model {
  3: num_classes: 90
  6: height: 320
  7: width: 320
  135: batch_size: 128
  166: num_steps: 50000
  165: fine_tune_checkpoint: "PATH_TO_BE_CONFIGURE

In [11]:
# Cek file yang berkaitan dengan training

from pathlib import Path

search_roots = [
    Path("/home/rzangef/SSD-PPE"),
    Path("/home/rzangef/spark-data/CHVG-SSD"),
]

print("=== PENCARIAN pipeline_ppe.config ===")
found = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("pipeline_ppe.config"):
            found.append(path)
            print(path)

print("\n=== PENCARIAN pipeline.config DI FOLDER TRAINING ===")
train_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

for path in train_dir.rglob("pipeline.config"):
    print(path)

if not found:
    print("\nTidak ditemukan file bernama pipeline_ppe.config.")

=== PENCARIAN pipeline_ppe.config ===
/home/rzangef/SSD-PPE/models/pipeline_ppe.config

=== PENCARIAN pipeline.config DI FOLDER TRAINING ===


In [12]:
from pathlib import Path

training_root = Path("/home/rzangef/SSD-PPE/training")

print("=== FILE KONFIGURASI DI FOLDER TRAINING ===")

for pattern in ["*.config", "*.pbtxt"]:
    for path in training_root.rglob(pattern):
        print(path)

=== FILE KONFIGURASI DI FOLDER TRAINING ===


In [13]:
# Periksa konfigurasi pipeline.config

from pathlib import Path
import re

config_path = Path(
    "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
)

print("File ditemukan:", config_path.exists())
print("Path:", config_path)

if config_path.exists():
    lines = config_path.read_text(errors="replace").splitlines()

    patterns = [
        r"^\s*model\s*\{",
        r"^\s*ssd\s*\{",
        r"^\s*num_classes\s*:",
        r"^\s*height\s*:",
        r"^\s*width\s*:",
        r"^\s*batch_size\s*:",
        r"^\s*num_steps\s*:",
        r"^\s*fine_tune_checkpoint\s*:",
        r"^\s*input_path\s*:",
        r"^\s*label_map_path\s*:",
        r"^\s*eval_config\s*\{",
        r"^\s*num_examples\s*:",
    ]

    for i, line in enumerate(lines, start=1):
        if any(re.search(p, line) for p in patterns):
            print(f"{i}: {line.strip()}")

File ditemukan: True
Path: /home/rzangef/SSD-PPE/models/pipeline_ppe.config
1: model {
2: ssd {
3: num_classes: 8
6: height: 320
7: width: 320
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps: 50000
175: label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
177: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/train.record"
180: eval_config {
185: label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/val.record"


In [14]:
# Evaluasi ulang checkpoint ckpt-24

import os
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)

SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

CONFIG_PATH = Path(
    "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
)

EVAL_OUTPUT_DIR = MODEL_DIR / "eval_ckpt24_verified"

# Periksa file yang dibutuhkan
print("Script tersedia       :", SCRIPT.exists())
print("Konfigurasi tersedia  :", CONFIG_PATH.exists())
print("Folder checkpoint     :", MODEL_DIR.exists())
print("Checkpoint ckpt-24    :", (MODEL_DIR / "ckpt-24.index").exists())

if not all([
    SCRIPT.exists(),
    CONFIG_PATH.exists(),
    MODEL_DIR.exists(),
    (MODEL_DIR / "ckpt-24.index").exists()
]):
    raise FileNotFoundError(
        "Ada file atau folder penting yang belum ditemukan."
    )

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(RESEARCH_DIR)
    + os.pathsep
    + env.get("PYTHONPATH", "")
).rstrip(os.pathsep)

cmd = [
    sys.executable,
    str(SCRIPT),
    f"--pipeline_config_path={CONFIG_PATH}",
    f"--model_dir={EVAL_OUTPUT_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=120",
    "--alsologtostderr",
]

print("\nPerintah evaluasi:")
print(" ".join(cmd))
print("\nEvaluasi dimulai...\n")

try:
    result = subprocess.run(
        cmd,
        cwd=str(RESEARCH_DIR),
        env=env,
        capture_output=True,
        text=True,
        timeout=600
    )

    print("Return code:", result.returncode)

    # Tampilkan bagian log yang relevan
    log = (result.stdout or "") + "\n" + (result.stderr or "")
    keywords = [
        "ckpt-24",
        "Evaluating",
        "DetectionBoxes",
        "Average Precision",
        "Average Recall",
        "ERROR",
        "Error",
        "Traceback",
        "NotFoundError",
        "Finished",
    ]

    matched = [
        line for line in log.splitlines()
        if any(word.lower() in line.lower() for word in keywords)
    ]

    print("\n=== LOG RELEVAN ===")
    print("\n".join(matched[-100:]) if matched else log[-12000:])

except subprocess.TimeoutExpired as exc:
    print("Proses melewati batas 600 detik.")
    print("Evaluasi belum tentu gagal; periksa log dan file output.")
    if exc.stdout:
        print(str(exc.stdout)[-6000:])
    if exc.stderr:
        print(str(exc.stderr)[-6000:])

Script tersedia       : True
Konfigurasi tersedia  : True
Folder checkpoint     : True
Checkpoint ckpt-24    : True

Perintah evaluasi:
/home/rzangef/SSD-PPE/.venv/bin/python /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py --pipeline_config_path=/home/rzangef/SSD-PPE/models/pipeline_ppe.config --model_dir=/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval_ckpt24_verified --checkpoint_dir=/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch --eval_timeout=120 --alsologtostderr

Evaluasi dimulai...

Return code: 0

=== LOG RELEVAN ===
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.243
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.479
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.206
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.009
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium |

In [1]:
# Cari Object detection Api

from pathlib import Path

PROJECT_DIR = Path("/home/rzangef/SSD-PPE")

search_roots = [
    PROJECT_DIR / "object-detection-api",
    PROJECT_DIR / "models",
    PROJECT_DIR / "spark-data",
]

print("=== MENCARI MODUL OBJECT DETECTION API ===")

for root in search_roots:
    print(f"\nMemeriksa: {root}")
    if not root.exists():
        print("Folder tidak ditemukan.")
        continue

    matches = list(root.glob("**/model_lib_v2.py"))
    if matches:
        for path in matches:
            print("Ditemukan:", path)
    else:
        print("model_lib_v2.py tidak ditemukan di lokasi ini.")

print("\n=== PEMERIKSAAN LOKASI YANG DIHARAPKAN ===")

expected = (
    PROJECT_DIR
    / "object-detection-api/research/object_detection/model_lib_v2.py"
)

print("Path:", expected)
print("Ada:", expected.exists())


=== MENCARI MODUL OBJECT DETECTION API ===

Memeriksa: /home/rzangef/SSD-PPE/object-detection-api
Ditemukan: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_lib_v2.py

Memeriksa: /home/rzangef/SSD-PPE/models
model_lib_v2.py tidak ditemukan di lokasi ini.

Memeriksa: /home/rzangef/SSD-PPE/spark-data
Folder tidak ditemukan.

=== PEMERIKSAAN LOKASI YANG DIHARAPKAN ===
Path: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_lib_v2.py
Ada: True


In [1]:
# Cek kesamaan dataset
from pathlib import Path
from collections import Counter
import tensorflow as tf
import hashlib

# ==========================================
# 1. LOKASI DATASET
# ==========================================
YOLO_TEST_DIR = Path(
    "/home/rzangef/spark-data/CHVG-YOLO/images/test"
)

SSD_TEST_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord/test.record"
)

print("=== PEMERIKSAAN LOKASI ===")
print("Folder test YOLO ada :", YOLO_TEST_DIR.is_dir())
print("Test TFRecord ada    :", SSD_TEST_RECORD.is_file())

if not YOLO_TEST_DIR.is_dir():
    raise FileNotFoundError(YOLO_TEST_DIR)

if not SSD_TEST_RECORD.is_file():
    raise FileNotFoundError(SSD_TEST_RECORD)


# ==========================================
# 2. DAFTAR GAMBAR YOLO
# ==========================================
extensions = {".jpg", ".jpeg", ".png", ".bmp"}
yolo_images = sorted(
    p for p in YOLO_TEST_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in extensions
)

yolo_names = [p.name.lower() for p in yolo_images]
yolo_stems = [p.stem.lower() for p in yolo_images]

print("\n=== DATA TEST YOLO ===")
print("Jumlah gambar :", len(yolo_images))
print("Jumlah nama unik:", len(set(yolo_names)))
print("Contoh nama:")
for p in yolo_images[:5]:
    print(" -", p.name)


# ==========================================
# 3. BACA NAMA DAN HASH GAMBAR DARI TFRECORD
# ==========================================
ssd_records = []

for raw_record in tf.data.TFRecordDataset(
    str(SSD_TEST_RECORD)
):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    features = example.features.feature

    def get_feature_bytes(key):
        feature = features.get(key)
        if feature and feature.bytes_list.value:
            return feature.bytes_list.value[0].decode(
                "utf-8", errors="replace"
            )
        return ""

    filename = get_feature_bytes("image/filename")
    source_id = get_feature_bytes("image/source_id")

    encoded = get_feature_bytes("image/encoded")
    # Untuk hash, gunakan bytes asli, bukan hasil decode teks.
    encoded_bytes = (
        features["image/encoded"].bytes_list.value[0]
        if features.get("image/encoded")
        and features["image/encoded"].bytes_list.value
        else b""
    )

    ssd_records.append({
        "filename": filename,
        "source_id": source_id,
        "stem": Path(filename or source_id).stem.lower(),
        "sha256": hashlib.sha256(encoded_bytes).hexdigest()
        if encoded_bytes else None
    })

ssd_stems = [r["stem"] for r in ssd_records]

print("\n=== DATA TEST SSD MOBILENET ===")
print("Jumlah record:", len(ssd_records))
print("Jumlah nama unik:", len(set(ssd_stems)))
print("Contoh nama:")
for r in ssd_records[:5]:
    print(
        " - filename:", repr(r["filename"]),
        "| source_id:", repr(r["source_id"])
    )


# ==========================================
# 4. BANDINGKAN NAMA GAMBAR
# ==========================================
yolo_set = set(yolo_stems)
ssd_set = set(ssd_stems)

only_yolo = sorted(yolo_set - ssd_set)
only_ssd = sorted(ssd_set - yolo_set)

print("\n=== HASIL PERBANDINGAN NAMA ===")
print("Jumlah nama unik YOLO :", len(yolo_set))
print("Jumlah nama unik SSD  :", len(ssd_set))
print("Nama hanya di YOLO    :", len(only_yolo))
print("Nama hanya di SSD     :", len(only_ssd))

print("\nContoh nama yang hanya di YOLO:")
print(only_yolo[:10])

print("\nContoh nama yang hanya di SSD:")
print(only_ssd[:10])

print("\nDuplikat nama YOLO:",
      [k for k, v in Counter(yolo_stems).items() if v > 1][:10])
print("Duplikat nama SSD:",
      [k for k, v in Counter(ssd_stems).items() if v > 1][:10])


# ==========================================
# 5. PERIKSA HASH GAMBAR
# ==========================================
yolo_hashes = {
    p.stem.lower(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in yolo_images
}

ssd_hashes = {
    r["stem"]: r["sha256"]
    for r in ssd_records
    if r["sha256"]
}

common_names = yolo_set & ssd_set
same_hash = [
    name for name in common_names
    if name in yolo_hashes
    and name in ssd_hashes
    and yolo_hashes[name] == ssd_hashes[name]
]
different_hash = [
    name for name in common_names
    if name in yolo_hashes
    and name in ssd_hashes
    and yolo_hashes[name] != ssd_hashes[name]
]

print("\n=== HASIL PEMERIKSAAN HASH ===")
print("Nama yang cocok :", len(common_names))
print("Hash identik    :", len(same_hash))
print("Hash berbeda    :", len(different_hash))
print("Hash tidak tersedia:", len(common_names) - len(same_hash) - len(different_hash))
print("Contoh hash berbeda:", different_hash[:10])

2026-10-20 08:49:02.372831: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 08:49:02.686069: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-20 08:49:02.686213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-20 08:49:02.748009: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-20 08:49:02.880354: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 08:49:02.882073: I tensorflow/core/platform/cpu_feature_guard.cc:1

=== PEMERIKSAAN LOKASI ===
Folder test YOLO ada : True
Test TFRecord ada    : True

=== DATA TEST YOLO ===
Jumlah gambar : 340
Jumlah nama unik: 340
Contoh nama:
 - 000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg
 - 000030_jpg.rf.39de95312521bfb83f176b4ee283a162.jpg
 - 000154_jpg.rf.c16424ee8936b0a863e3f4dcccdcbdb8.jpg
 - 000221_jpg.rf.92483e3414819358a5aa335d941e80a3.jpg
 - 000245_jpg.rf.3bfad5696b7411665e272b0bec4f1907.jpg

=== DATA TEST SSD MOBILENET ===
Jumlah record: 340
Jumlah nama unik: 340
Contoh nama:
 - filename: '000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa.jpg' | source_id: '000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa.jpg'
 - filename: '000091_jpg.rf.75339227b3c4f06e011096c7fa4e1ff5.jpg' | source_id: '000091_jpg.rf.75339227b3c4f06e011096c7fa4e1ff5.jpg'
 - filename: '000132_jpg.rf.39acd7bca3d210ca59deafbe0204b1d8.jpg' | source_id: '000132_jpg.rf.39acd7bca3d210ca59deafbe0204b1d8.jpg'
 - filename: '000240_jpg.rf.0b240cbf834bb8178f6355b21ab71679.jpg' | source_id: '00024

In [2]:
# Periksa Anotasi gambar data test pada dataset ssd karena hanya 68 gambar yang sama dengan model yolo
from pathlib import Path
from collections import defaultdict
import tensorflow as tf
import numpy as np

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
YOLO_IMG_DIR = YOLO_ROOT / "images/test"
YOLO_LABEL_DIR = YOLO_ROOT / "labels/test"

SSD_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord/test.record"
)
LABEL_MAP = Path(
    "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
)

# 1. Baca daftar label kelas SSD
print("=== LABEL MAP SSD ===")
print(LABEL_MAP.read_text())

# 2. Baca nama gambar dari YOLO
yolo_images = {
    p.stem.lower(): p
    for p in YOLO_IMG_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
}

# 3. Baca label YOLO
yolo_labels = {}
for stem in yolo_images:
    label_path = YOLO_LABEL_DIR / f"{stem}.txt"
    boxes = []

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
                boxes.append((cls, xc, yc, w, h))

    yolo_labels[stem] = boxes

# 4. Baca anotasi TFRecord SSD
ssd_examples = {}

for raw in tf.data.TFRecordDataset(str(SSD_RECORD)):
    ex = tf.train.Example()
    ex.ParseFromString(raw.numpy())
    f = ex.features.feature

    def get_bytes(key):
        vals = f[key].bytes_list.value
        return vals[0].decode("utf-8", errors="replace") if vals else ""

    filename = get_bytes("image/filename")
    source_id = get_bytes("image/source_id")
    stem = Path(filename or source_id).stem.lower()

    xmin = list(f["image/object/bbox/xmin"].float_list.value)
    xmax = list(f["image/object/bbox/xmax"].float_list.value)
    ymin = list(f["image/object/bbox/ymin"].float_list.value)
    ymax = list(f["image/object/bbox/ymax"].float_list.value)
    classes = list(f["image/object/class/label"].int64_list.value)

    boxes = []
    for i in range(min(
        len(xmin), len(xmax), len(ymin), len(ymax), len(classes)
    )):
        xc = (xmin[i] + xmax[i]) / 2
        yc = (ymin[i] + ymax[i]) / 2
        w = xmax[i] - xmin[i]
        h = ymax[i] - ymin[i]

        # TensorFlow Object Detection API umumnya memakai ID kelas 1-based,
        # sedangkan label YOLO memakai ID 0-based.
        boxes.append((classes[i] - 1, xc, yc, w, h))

    ssd_examples[stem] = {
        "filename": filename,
        "boxes": boxes
    }

# 5. Bandingkan anotasi pada gambar yang sama
common = sorted(set(yolo_images) & set(ssd_examples))
exact_matches = []
class_count_matches = []
mismatches = []

for stem in common:
    y_boxes = yolo_labels.get(stem, [])
    s_boxes = ssd_examples[stem]["boxes"]

    y_sorted = sorted(y_boxes, key=lambda b: (b[0], *b[1:]))
    s_sorted = sorted(s_boxes, key=lambda b: (b[0], *b[1:]))

    same_classes = sorted(b[0] for b in y_sorted) == sorted(
        b[0] for b in s_sorted
    )

    # Toleransi 0,002 untuk perbedaan pembulatan koordinat
    same_boxes = (
        len(y_sorted) == len(s_sorted)
        and same_classes
        and all(
            a[0] == b[0]
            and np.allclose(a[1:], b[1:], atol=0.002, rtol=0)
            for a, b in zip(y_sorted, s_sorted)
        )
    )

    if same_boxes:
        exact_matches.append(stem)
    elif same_classes and len(y_sorted) == len(s_sorted):
        class_count_matches.append(stem)
        mismatches.append(stem)
    else:
        mismatches.append(stem)

print("\n=== HASIL PEMERIKSAAN ANOTASI ===")
print("Jumlah gambar yang sama:", len(common))
print("Anotasi cocok:", len(exact_matches))
print("Jumlah/kelas cocok, koordinat perlu dicek:",
      len(class_count_matches))
print("Anotasi berbeda/perlu diperiksa:", len(mismatches))
print("\nContoh anotasi yang perlu diperiksa:", mismatches[:10])
print("\nContoh gambar dengan anotasi cocok:", exact_matches[:10])

=== LABEL MAP SSD ===
item {
    id: 1
    name: 'person'
}

item {
    id: 2
    name: 'vest'
}

item {
    id: 3
    name: 'yellow'
}

item {
    id: 4
    name: 'white'
}

item {
    id: 5
    name: 'glass'
}

item {
    id: 6
    name: 'blue'
}

item {
    id: 7
    name: 'red'
}

item {
    id: 8
    name: 'head'
}



=== HASIL PEMERIKSAAN ANOTASI ===
Jumlah gambar yang sama: 68
Anotasi cocok: 65
Jumlah/kelas cocok, koordinat perlu dicek: 0
Anotasi berbeda/perlu diperiksa: 3

Contoh anotasi yang perlu diperiksa: ['lennard-kollossa-lxn9o5cor1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07', 'mufid-majnun-bkowuu27l5w-unsplash_jpg.rf.8cba52a29ed45344cd2bde655fc29acd', 'ppe_1349_jpg.rf.55b4dc80bbd369335bda973b07d58b3e']

Contoh gambar dengan anotasi cocok: ['000671_jpg.rf.6efdcdb3b980e4de8e510007863f2138', 'gettyimages-1047616916-612x612_jpg.rf.4ec6cb2367d950a93de502f6ebcbead4', 'gettyimages-1130648954-612x612_jpg.rf.b0328dbfe6d27fed005334b044353556', 'gettyimages-1199683614-612x61

In [3]:
# Cek anotasi 3 gambar yang berbeda
from pathlib import Path
import tensorflow as tf
import numpy as np

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
YOLO_LABEL_DIR = YOLO_ROOT / "labels/test"
SSD_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord/test.record"
)

target_names = {
    "lennard-kollossa-lxn9o5cor1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07",
    "mufid-majnun-bkowuu27l5w-unsplash_jpg.rf.8cba52a29ed45344cd2bde655fc29acd",
    "ppe_1349_jpg.rf.55b4dc80bbd369335bda973b07d58b3e",
}

class_names = {
    0: "person", 1: "vest", 2: "yellow", 3: "white",
    4: "glass", 5: "blue", 6: "red", 7: "head"
}

# Baca anotasi YOLO
yolo_data = {}

for stem in target_names:
    path = YOLO_LABEL_DIR / f"{stem}.txt"
    boxes = []

    if path.exists():
        for line in path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                cls = int(parts[0])
                coords = list(map(float, parts[1:5]))
                boxes.append((cls, coords))

    yolo_data[stem] = {
        "exists": path.exists(),
        "boxes": boxes
    }

# Baca anotasi dari TFRecord SSD
ssd_data = {}

for raw in tf.data.TFRecordDataset(str(SSD_RECORD)):
    ex = tf.train.Example()
    ex.ParseFromString(raw.numpy())
    f = ex.features.feature

    def get_bytes(key):
        values = f[key].bytes_list.value
        return values[0].decode("utf-8", errors="replace") if values else ""

    filename = get_bytes("image/filename")
    source_id = get_bytes("image/source_id")
    stem = Path(filename or source_id).stem.lower()

    if stem not in target_names:
        continue

    xmin = list(f["image/object/bbox/xmin"].float_list.value)
    xmax = list(f["image/object/bbox/xmax"].float_list.value)
    ymin = list(f["image/object/bbox/ymin"].float_list.value)
    ymax = list(f["image/object/bbox/ymax"].float_list.value)
    classes = list(f["image/object/class/label"].int64_list.value)

    boxes = []
    for i in range(min(len(xmin), len(xmax), len(ymin), len(ymax), len(classes))):
        coords = [
            (xmin[i] + xmax[i]) / 2,
            (ymin[i] + ymax[i]) / 2,
            xmax[i] - xmin[i],
            ymax[i] - ymin[i]
        ]
        boxes.append((int(classes[i]) - 1, coords))

    ssd_data[stem] = boxes

# Cetak perbandingan
print("=== DETAIL ANOTASI 3 GAMBAR ===")

for stem in sorted(target_names):
    print("\n" + "=" * 75)
    print("Gambar:", stem)

    y = yolo_data[stem]
    s = ssd_data.get(stem)

    print("File label YOLO tersedia:", y["exists"])
    print("Jumlah objek YOLO:", len(y["boxes"]))
    print("Jumlah objek SSD :", len(s) if s is not None else "RECORD TIDAK DITEMUKAN")

    print("\nYOLO (class, x_center, y_center, width, height):")
    for cls, coords in sorted(y["boxes"], key=lambda x: (x[0], x[1])):
        print(class_names.get(cls, f"unknown_{cls}"),
              [round(v, 5) for v in coords])

    print("\nSSD (class, x_center, y_center, width, height):")
    if s is not None:
        for cls, coords in sorted(s, key=lambda x: (x[0], x[1])):
            print(class_names.get(cls, f"unknown_{cls}"),
                  [round(v, 5) for v in coords])

    print("\nJumlah objek per kelas:")
    for cls, name in class_names.items():
        yc = sum(1 for c, _ in y["boxes"] if c == cls)
        sc = sum(1 for c, _ in (s or []) if c == cls)
        if yc or sc:
            print(f"{name:8s} | YOLO={yc} | SSD={sc}")

=== DETAIL ANOTASI 3 GAMBAR ===

Gambar: lennard-kollossa-lxn9o5cor1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07
File label YOLO tersedia: False
Jumlah objek YOLO: 0
Jumlah objek SSD : 6

YOLO (class, x_center, y_center, width, height):

SSD (class, x_center, y_center, width, height):
person [0.28047, 0.58125, 0.35156, 0.8375]
person [0.55937, 0.61406, 0.19687, 0.77187]
vest [0.27109, 0.71094, 0.28281, 0.55313]
vest [0.55703, 0.55547, 0.16719, 0.31406]
yellow [0.30781, 0.25469, 0.15937, 0.15937]
yellow [0.55547, 0.29141, 0.10156, 0.10156]

Jumlah objek per kelas:
person   | YOLO=0 | SSD=2
vest     | YOLO=0 | SSD=2
yellow   | YOLO=0 | SSD=2

Gambar: mufid-majnun-bkowuu27l5w-unsplash_jpg.rf.8cba52a29ed45344cd2bde655fc29acd
File label YOLO tersedia: False
Jumlah objek YOLO: 0
Jumlah objek SSD : 17

YOLO (class, x_center, y_center, width, height):

SSD (class, x_center, y_center, width, height):
person [0.11953, 0.61094, 0.16719, 0.75625]
person [0.26953, 0.61797, 0.18594, 0.73906]
p

In [4]:

from pathlib import Path

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
SOURCE_ROOT = Path(
    "/home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset"
)

targets = [
    "lennard-kollossa-lxn9o5cor1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07",
    "mufid-majnun-bkowuu27l5w-unsplash_jpg.rf.8cba52a29ed45344cd2bde655fc29acd",
    "ppe_1349_jpg.rf.55b4dc80bbd369335bda973b07d58b3e",
]

image_exts = {".jpg", ".jpeg", ".png", ".bmp"}

# Buat indeks file berdasarkan nama dasar (stem)
def build_index(root, extensions):
    index = {}
    if not root.exists():
        return index

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in extensions:
            index.setdefault(p.stem.lower(), []).append(p)

    return index

yolo_images_all = build_index(YOLO_ROOT, image_exts)
yolo_labels_all = build_index(YOLO_ROOT, {".txt"})
source_images_all = build_index(SOURCE_ROOT, image_exts)
source_xml_all = build_index(SOURCE_ROOT, {".xml"})

print("=== STATUS FOLDER ===")
print("YOLO root ada   :", YOLO_ROOT.exists())
print("Source root ada :", SOURCE_ROOT.exists())

for stem in targets:
    print("\n" + "=" * 80)
    print("TARGET:", stem)

    checks = [
        ("Gambar dalam YOLO", yolo_images_all),
        ("Label TXT dalam YOLO", yolo_labels_all),
        ("Gambar sumber", source_images_all),
        ("Anotasi XML sumber", source_xml_all),
    ]

    for label, index in checks:
        matches = index.get(stem.lower(), [])
        print(f"\n{label}: {len(matches)} file")
        for p in matches[:10]:
            print(" -", p)

            # Tampilkan isi file label YOLO jika ditemukan
            if label == "Label TXT dalam YOLO":
                try:
                    print("   Isi:", repr(p.read_text()[:1500]))
                except Exception as e:
                    print("   Gagal membaca:", e)

            # Tampilkan ringkasan XML jika ditemukan
            if label == "Anotasi XML sumber":
                try:
                    import xml.etree.ElementTree as ET
                    root = ET.parse(p).getroot()
                    objects = root.findall("object")
                    classes = [
                        obj.findtext("name", default="unknown")
                        for obj in objects
                    ]
                    print("   Jumlah objek XML:", len(objects))
                    print("   Kelas XML:", classes)
                except Exception as e:
                    print("   Gagal membaca XML:", e)

=== STATUS FOLDER ===
YOLO root ada   : True
Source root ada : True

TARGET: lennard-kollossa-lxn9o5cor1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07

Gambar dalam YOLO: 1 file
 - /home/rzangef/spark-data/CHVG-YOLO/images/test/lennard-kollossa-lXN9O5cOr1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07.jpg

Label TXT dalam YOLO: 2 file
 - /home/rzangef/spark-data/CHVG-YOLO/labels/test/lennard-kollossa-lXN9O5cOr1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07.txt
   Isi: '2 0.307812 0.254688 0.159375 0.159375\n2 0.555469 0.291406 0.101562 0.101562\n1 0.557031 0.555469 0.167187 0.314063\n0 0.280469 0.581250 0.351562 0.837500\n0 0.559375 0.614062 0.196875 0.771875\n1 0.271094 0.710938 0.282813 0.553125'
 - /home/rzangef/spark-data/CHVG-YOLO/labels/all/lennard-kollossa-lXN9O5cOr1s-unsplash_jpg.rf.a15aea04e084d8efb9b3b0eb65cf6b07.txt
   Isi: '2 0.307812 0.254688 0.159375 0.159375\n2 0.555469 0.291406 0.101562 0.101562\n1 0.557031 0.555469 0.167187 0.314063\n0 0.280469 0.581250 0.35

In [5]:

from pathlib import Path
import tensorflow as tf
import numpy as np

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
YOLO_LABEL_DIR = YOLO_ROOT / "labels/test"
SSD_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord/test.record"
)

# Indeks label YOLO tanpa membedakan huruf besar/kecil
yolo_label_index = {
    p.stem.lower(): p
    for p in YOLO_LABEL_DIR.glob("*.txt")
}

# Baca label SSD dari TFRecord
ssd_annotations = {}

for raw in tf.data.TFRecordDataset(str(SSD_RECORD)):
    ex = tf.train.Example()
    ex.ParseFromString(raw.numpy())
    f = ex.features.feature

    def get_bytes(key):
        vals = f[key].bytes_list.value
        return vals[0].decode("utf-8", errors="replace") if vals else ""

    filename = get_bytes("image/filename")
    source_id = get_bytes("image/source_id")
    stem = Path(filename or source_id).stem.lower()

    xmin = list(f["image/object/bbox/xmin"].float_list.value)
    xmax = list(f["image/object/bbox/xmax"].float_list.value)
    ymin = list(f["image/object/bbox/ymin"].float_list.value)
    ymax = list(f["image/object/bbox/ymax"].float_list.value)
    classes = list(f["image/object/class/label"].int64_list.value)

    boxes = []
    for i in range(min(len(xmin), len(xmax), len(ymin), len(ymax), len(classes))):
        boxes.append((
            int(classes[i]) - 1,
            (xmin[i] + xmax[i]) / 2,
            (ymin[i] + ymax[i]) / 2,
            xmax[i] - xmin[i],
            ymax[i] - ymin[i]
        ))

    ssd_annotations[stem] = boxes

# Bandingkan anotasi semua nama gambar yang sama
common = sorted(set(yolo_label_index) & set(ssd_annotations))
missing_label = []
exact_match = []
different = []

for stem in common:
    path = yolo_label_index[stem]
    yolo_boxes = []

    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            cls = int(parts[0])
            coords = tuple(map(float, parts[1:5]))
            yolo_boxes.append((cls, *coords))

    ssd_boxes = ssd_annotations[stem]

    # Urutkan berdasarkan kelas dan koordinat untuk perbandingan stabil
    yolo_sorted = sorted(yolo_boxes)
    ssd_sorted = sorted(ssd_boxes)

    same = (
        len(yolo_sorted) == len(ssd_sorted)
        and all(
            a[0] == b[0]
            and np.allclose(a[1:], b[1:], atol=0.002, rtol=0)
            for a, b in zip(yolo_sorted, ssd_sorted)
        )
    )

    if same:
        exact_match.append(stem)
    else:
        different.append(stem)

# Gambar SSD yang tidak ditemukan labelnya pada folder label/test YOLO
for stem in sorted(set(ssd_annotations) - set(yolo_label_index)):
    missing_label.append(stem)

print("=== HASIL PEMERIKSAAN ULANG ===")
print("Jumlah gambar bersama:", len(common))
print("Anotasi cocok:", len(exact_match))
print("Anotasi berbeda:", len(different))
print("Label YOLO tidak ditemukan untuk gambar SSD:",
      len(missing_label))

print("\nContoh anotasi berbeda:", different[:15])
print("\nContoh label yang tidak ditemukan:", missing_label[:15])

=== HASIL PEMERIKSAAN ULANG ===
Jumlah gambar bersama: 68
Anotasi cocok: 68
Anotasi berbeda: 0
Label YOLO tidak ditemukan untuk gambar SSD: 272

Contoh anotasi berbeda: []

Contoh label yang tidak ditemukan: ['000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa', '000091_jpg.rf.75339227b3c4f06e011096c7fa4e1ff5', '000132_jpg.rf.39acd7bca3d210ca59deafbe0204b1d8', '000240_jpg.rf.0b240cbf834bb8178f6355b21ab71679', '000242_jpg.rf.1b2d022564037ada01201c2a64aaa35f', '000570_jpg.rf.9833d6e0dea614a716e6a792937a2f5c', '000915_jpg.rf.b9171f77e9e345c57d636405433dc295', '001056_jpg.rf.a8ef22a105f1518bafe30b7215c355d0', '001244_jpg.rf.b9d0c29dab3977371f8707e4f8cc407b', '001360_jpg.rf.7ab4dce21523a37413c2f729fda68597', '001500_jpg.rf.b92553164f7dac78fb89761e3966c69e', '002622_jpg.rf.882c5edfc64aa5863427a010ad0c44cd', '002673_jpg.rf.960c82234d2efce92759432916a12887', 'ascel-kadhem-umn4sjz7njg-unsplash_jpg.rf.6f18f329d6253796f13139f9c8a58817', 'frank-albrecht-poh0xf9zaye-unsplash_jpg.rf.475871208fafb4484016

In [1]:
# Cek apakah tfrecord dibuat dari dataset SSD-PPE/CHVG-Dataset-SSD
from pathlib import Path
import tensorflow as tf

SSD_ROOT = Path(
    "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD"
)
TFRECORD_ROOT = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord"
)

# Kumpulkan nama file gambar pada folder test SSD
ssd_test_dir = SSD_ROOT / "test"

jpg_extensions = {".jpg", ".jpeg", ".png"}
ssd_test_images = {
    p.stem.lower()
    for p in ssd_test_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in jpg_extensions
}

# Kumpulkan nama gambar dari test.record
record_test_images = set()
record_count = 0
missing_filename_count = 0

for raw in tf.data.TFRecordDataset(
    str(TFRECORD_ROOT / "test.record")
):
    ex = tf.train.Example()
    ex.ParseFromString(raw.numpy())
    features = ex.features.feature

    def get_string(key):
        values = features[key].bytes_list.value
        return (
            values[0].decode("utf-8", errors="replace")
            if values else ""
        )

    filename = get_string("image/filename")
    source_id = get_string("image/source_id")
    image_name = filename or source_id

    record_count += 1

    if image_name:
        record_test_images.add(
            Path(image_name).stem.lower()
        )
    else:
        missing_filename_count += 1

# Bandingkan kedua sumber
common = ssd_test_images & record_test_images
only_folder = ssd_test_images - record_test_images
only_record = record_test_images - ssd_test_images

print("=== PEMERIKSAAN TEST SSD ===")
print("Jumlah gambar JPG/XML di folder test :", len(ssd_test_images))
print("Jumlah record di test.record         :", record_count)
print("Nama gambar unik di test.record      :", len(record_test_images))
print("Nama gambar yang cocok               :", len(common))
print("Hanya ada di folder test             :", len(only_folder))
print("Hanya ada di test.record             :", len(only_record))
print("Record tanpa nama gambar             :", missing_filename_count)

print("\nContoh hanya di folder test:")
print(sorted(only_folder)[:10])

print("\nContoh hanya di test.record:")
print(sorted(only_record)[:10])

2026-10-20 10:36:33.938124: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 10:36:34.644818: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-20 10:36:34.645145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-20 10:36:34.782151: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-20 10:36:35.071583: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 10:36:35.076345: I tensorflow/core/platform/cpu_feature_guard.cc:1

=== PEMERIKSAAN TEST SSD ===
Jumlah gambar JPG/XML di folder test : 340
Jumlah record di test.record         : 340
Nama gambar unik di test.record      : 340
Nama gambar yang cocok               : 340
Hanya ada di folder test             : 0
Hanya ada di test.record             : 0
Record tanpa nama gambar             : 0

Contoh hanya di folder test:
[]

Contoh hanya di test.record:
[]


In [2]:
# Cek test SDD = tes YOLO ?
from pathlib import Path

SSD_TEST = Path(
    "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/test"
)
YOLO_TEST = Path(
    "/home/rzangef/spark-data/CHVG-YOLO/images/test"
)

EXTENSIONS = {".jpg", ".jpeg", ".png"}

def image_index(folder):
    return {
        p.stem.lower(): p
        for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    }

ssd_images = image_index(SSD_TEST)
yolo_images = image_index(YOLO_TEST)

ssd_names = set(ssd_images)
yolo_names = set(yolo_images)

common = ssd_names & yolo_names
ssd_only = ssd_names - yolo_names
yolo_only = yolo_names - ssd_names

print("=== PERBANDINGAN FOLDER TEST SSD DAN YOLO ===")
print("Jumlah gambar SSD test :", len(ssd_names))
print("Jumlah gambar YOLO test:", len(yolo_names))
print("Gambar dengan nama sama:", len(common))
print("Hanya di SSD test      :", len(ssd_only))
print("Hanya di YOLO test     :", len(yolo_only))

# Pastikan gambar yang bernama sama benar-benar identik
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

identical = sum(
    sha256(ssd_images[name]) == sha256(yolo_images[name])
    for name in common
)

print("File gambar identik    :", identical)
print("File gambar berbeda    :", len(common) - identical)

print("\nContoh gambar khusus SSD:", sorted(ssd_only)[:10])
print("Contoh gambar khusus YOLO:", sorted(yolo_only)[:10])

=== PERBANDINGAN FOLDER TEST SSD DAN YOLO ===
Jumlah gambar SSD test : 340
Jumlah gambar YOLO test: 340
Gambar dengan nama sama: 68
Hanya di SSD test      : 272
Hanya di YOLO test     : 272
File gambar identik    : 68
File gambar berbeda    : 0

Contoh gambar khusus SSD: ['000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa', '000091_jpg.rf.75339227b3c4f06e011096c7fa4e1ff5', '000132_jpg.rf.39acd7bca3d210ca59deafbe0204b1d8', '000240_jpg.rf.0b240cbf834bb8178f6355b21ab71679', '000242_jpg.rf.1b2d022564037ada01201c2a64aaa35f', '000570_jpg.rf.9833d6e0dea614a716e6a792937a2f5c', '000915_jpg.rf.b9171f77e9e345c57d636405433dc295', '001056_jpg.rf.a8ef22a105f1518bafe30b7215c355d0', '001244_jpg.rf.b9d0c29dab3977371f8707e4f8cc407b', '001360_jpg.rf.7ab4dce21523a37413c2f729fda68597']
Contoh gambar khusus YOLO: ['000019_jpg.rf.549923f46def56cf8c294def971cc7d9', '000030_jpg.rf.39de95312521bfb83f176b4ee283a162', '000154_jpg.rf.c16424ee8936b0a863e3f4dcccdcbdb8', '000221_jpg.rf.92483e3414819358a5aa335d941e80a3',

In [3]:

from pathlib import Path
import shutil
import sys

# ==================================================
# 1. KONFIGURASI PATH
# ==================================================
YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")

XML_ROOT = Path(
    "/home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset"
)

SSD_DATASET = Path(
    "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD"
)

OUTPUT_DIR = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split"
)

SPLITS = ["train", "val", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# ==================================================
# 2. PERIKSA PYTHON DAN SUMBER DATA
# ==================================================
print("=== ENVIRONMENT ===")
print("Python executable :", sys.executable)
print("YOLO dataset      :", YOLO_ROOT)
print("XML source        :", XML_ROOT)
print("SSD dataset       :", SSD_DATASET)
print("Output directory  :", OUTPUT_DIR)

required_paths = {
    "YOLO root": YOLO_ROOT,
    "XML source": XML_ROOT,
    "SSD dataset": SSD_DATASET,
    "YOLO train images": YOLO_ROOT / "images/train",
    "YOLO val images": YOLO_ROOT / "images/val",
    "YOLO test images": YOLO_ROOT / "images/test",
    "Label map": SSD_DATASET / "label_map.pbtxt",
}

print("\n=== PEMERIKSAAN PATH ===")
for name, path in required_paths.items():
    print(f"{name:22s}: {'ADA' if path.exists() else 'TIDAK ADA'}")

if not all(p.exists() for p in required_paths.values()):
    raise FileNotFoundError(
        "Ada sumber yang tidak ditemukan. Periksa path di atas."
    )

# ==================================================
# 3. HITUNG GAMBAR PADA SETIAP SPLIT
# ==================================================
print("\n=== JUMLAH GAMBAR YOLO ===")

for split in SPLITS:
    folder = YOLO_ROOT / "images" / split
    images = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    print(f"{split:6s}: {len(images)} gambar")

# ==================================================
# 4. BUAT FOLDER OUTPUT TERPISAH
# ==================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jangan menimpa label map yang sudah ada.
source_label_map = SSD_DATASET / "label_map.pbtxt"
target_label_map = OUTPUT_DIR / "label_map.pbtxt"

if target_label_map.exists():
    print("\nLabel map tujuan sudah ada; tidak ditimpa.")
else:
    shutil.copy2(source_label_map, target_label_map)
    print("\nLabel map berhasil disalin ke:")
    print(target_label_map)

print("\n=== ISI FOLDER OUTPUT SAAT INI ===")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)

=== ENVIRONMENT ===
Python executable : /home/rzangef/SSD-PPE/.venv/bin/python
YOLO dataset      : /home/rzangef/spark-data/CHVG-YOLO
XML source        : /home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset
SSD dataset       : /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD
Output directory  : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split

=== PEMERIKSAAN PATH ===
YOLO root             : ADA
XML source            : ADA
SSD dataset           : ADA
YOLO train images     : ADA
YOLO val images       : ADA
YOLO test images      : ADA
Label map             : ADA

=== JUMLAH GAMBAR YOLO ===
train : 1189 gambar
val   : 170 gambar
test  : 340 gambar

Label map berhasil disalin ke:
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt

=== ISI FOLDER OUTPUT SAAT INI ===
label_map.pbtxt


In [4]:

from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
XML_ROOT = Path(
    "/home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset"
)

EXPECTED_CLASSES = {
    "person", "vest", "yellow", "white",
    "glass", "blue", "red", "head"
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Buat indeks XML agar pencarian lebih cepat
xml_index = {}

for xml_path in XML_ROOT.rglob("*.xml"):
    xml_index.setdefault(xml_path.stem.lower(), []).append(xml_path)

print("Jumlah file XML yang ditemukan:", sum(map(len, xml_index.values())))
print("Nama dasar XML unik:", len(xml_index))

for split in ["train", "val", "test"]:
    image_dir = YOLO_ROOT / "images" / split

    images = [
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    missing_xml = []
    duplicate_xml = []
    unreadable_xml = []
    unknown_classes = Counter()
    empty_xml = []
    object_counts = Counter()

    for image_path in images:
        matches = xml_index.get(image_path.stem.lower(), [])

        if len(matches) == 0:
            missing_xml.append(image_path.name)
            continue

        if len(matches) > 1:
            duplicate_xml.append(image_path.name)
            continue

        xml_path = matches[0]

        try:
            root = ET.parse(xml_path).getroot()
        except Exception:
            unreadable_xml.append(xml_path.name)
            continue

        objects = root.findall("object")

        if not objects:
            empty_xml.append(xml_path.name)

        for obj in objects:
            name = (obj.findtext("name") or "").strip().lower()
            object_counts[name] += 1

            if name not in EXPECTED_CLASSES:
                unknown_classes[name] += 1

    print(f"\n=== {split.upper()} ===")
    print("Jumlah gambar          :", len(images))
    print("XML tidak ditemukan    :", len(missing_xml))
    print("XML dengan nama ambigu :", len(duplicate_xml))
    print("XML tidak dapat dibaca :", len(unreadable_xml))
    print("XML tanpa objek        :", len(empty_xml))
    print("Kelas di luar daftar   :", dict(unknown_classes))

    if missing_xml:
        print("Contoh gambar tanpa XML:", missing_xml[:5])
    if duplicate_xml:
        print("Contoh pasangan ambigu :", duplicate_xml[:5])
    if unreadable_xml:
        print("Contoh XML bermasalah  :", unreadable_xml[:5])

    print("Jumlah objek per kelas :", dict(sorted(object_counts.items())))

Jumlah file XML yang ditemukan: 1699
Nama dasar XML unik: 1699

=== TRAIN ===
Jumlah gambar          : 1189
XML tidak ditemukan    : 0
XML dengan nama ambigu : 0
XML tidak dapat dibaca : 0
XML tanpa objek        : 1
Kelas di luar daftar   : {}
Jumlah objek per kelas : {'blue': 343, 'glass': 380, 'head': 487, 'person': 3220, 'red': 364, 'vest': 1474, 'white': 899, 'yellow': 841}

=== VAL ===
Jumlah gambar          : 170
XML tidak ditemukan    : 0
XML dengan nama ambigu : 0
XML tidak dapat dibaca : 0
XML tanpa objek        : 0
Kelas di luar daftar   : {}
Jumlah objek per kelas : {'blue': 55, 'glass': 49, 'head': 92, 'person': 490, 'red': 50, 'vest': 233, 'white': 130, 'yellow': 112}

=== TEST ===
Jumlah gambar          : 340
XML tidak ditemukan    : 0
XML dengan nama ambigu : 0
XML tidak dapat dibaca : 0
XML tanpa objek        : 0
Kelas di luar daftar   : {}
Jumlah objek per kelas : {'blue': 145, 'glass': 103, 'head': 151, 'person': 964, 'red': 82, 'vest': 430, 'white': 263, 'yellow': 24

In [5]:
# Cek label.map
from pathlib import Path
import re

LABEL_MAP = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
)

text = LABEL_MAP.read_text(encoding="utf-8")

# Baca item { id: ... name: ... } dari label map
items = re.findall(
    r"item\s*\{(.*?)\}",
    text,
    flags=re.DOTALL
)

class_map = {}

for item in items:
    id_match = re.search(r"\bid\s*:\s*(\d+)", item)
    name_match = re.search(r"\bname\s*:\s*['\"]([^'\"]+)['\"]", item)

    if id_match and name_match:
        class_id = int(id_match.group(1))
        class_name = name_match.group(1).strip().lower()
        class_map[class_id] = class_name

print("=== LABEL MAP SSD ===")
for class_id, class_name in sorted(class_map.items()):
    print(f"{class_id}: {class_name}")

expected = {
    "person", "vest", "yellow", "white",
    "glass", "blue", "red", "head"
}

print("\nJumlah kelas:", len(class_map))
print("Semua kelas sesuai:", set(class_map.values()) == expected)
print("ID kelas berurutan mulai 1:",
      sorted(class_map.keys()) == list(range(1, 9)))

=== LABEL MAP SSD ===
1: person
2: vest
3: yellow
4: white
5: glass
6: blue
7: red
8: head

Jumlah kelas: 8
Semua kelas sesuai: True
ID kelas berurutan mulai 1: True


In [6]:
# Cek konfigurasi lama

from pathlib import Path
import re

PROJECT = Path("/home/rzangef/SSD-PPE")

# Konfigurasi SSD lama
OLD_CONFIG = PROJECT / "models/pipeline_ppe.config"

# Folder checkpoint SSD hasil eksperimen sebelumnya
OLD_MODEL_DIR = (
    PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

# TFRecord baru yang sudah divalidasi
NEW_DATA_DIR = PROJECT / "dataset/tfrecord_yolo_split"

print("=== PEMERIKSAAN FILE ===")
print("Konfigurasi lama :", OLD_CONFIG, OLD_CONFIG.exists())
print("Folder checkpoint:", OLD_MODEL_DIR, OLD_MODEL_DIR.exists())
print("Train record     :", (NEW_DATA_DIR / "train.record").exists())
print("Val record       :", (NEW_DATA_DIR / "val.record").exists())
print("Test record      :", (NEW_DATA_DIR / "test.record").exists())
print("Label map        :", (NEW_DATA_DIR / "label_map.pbtxt").exists())

if not OLD_CONFIG.exists():
    raise FileNotFoundError(
        f"Konfigurasi tidak ditemukan: {OLD_CONFIG}"
    )

config_text = OLD_CONFIG.read_text(encoding="utf-8")

print("\n=== PENGATURAN PENTING DI CONFIG ===")

patterns = [
    r"num_classes\s*:\s*\d+",
    r"batch_size\s*:\s*\d+",
    r"num_steps\s*:\s*\d+",
    r"fine_tune_checkpoint\s*:\s*['\"][^'\"]+['\"]",
    r"fine_tune_checkpoint_type\s*:\s*['\"][^'\"]+['\"]",
    r"num_epochs\s*:\s*\d+",
    r"input_path\s*:\s*['\"][^'\"]+['\"]",
    r"label_map_path\s*:\s*['\"][^'\"]+['\"]",
]

for pattern in patterns:
    matches = re.findall(pattern, config_text)
    for match in matches:
        print(match)

print("\n=== CHECKPOINT LAMA ===")
if OLD_MODEL_DIR.exists():
    for path in sorted(OLD_MODEL_DIR.glob("ckpt-*.index")):
        print(path.name)
else:
    print("Folder checkpoint lama tidak ditemukan.")

=== PEMERIKSAAN FILE ===
Konfigurasi lama : /home/rzangef/SSD-PPE/models/pipeline_ppe.config True
Folder checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch True
Train record     : True
Val record       : True
Test record      : True
Label map        : True

=== PENGATURAN PENTING DI CONFIG ===
num_classes: 8
batch_size: 1
num_steps: 50000
fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
fine_tune_checkpoint_type: "detection"
num_epochs: 1
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/train.record"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord/val.record"
label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"
label_map_path: "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt"

=== CHECKPOINT LAMA ===
ckpt-18.index
ckpt-19.index
ckpt-20.index
ckpt-21.index
ckpt-22.index
ckpt-23.index
ckpt-24.index


In [7]:
# Cek kondisi laptop

import os
import shutil
import subprocess
from pathlib import Path

print("=== 1. SISTEM DAN CPU ===")
print("Jumlah CPU logis:", os.cpu_count())

try:
    import psutil

    mem = psutil.virtual_memory()
    print(f"RAM total       : {mem.total / 1024**3:.2f} GB")
    print(f"RAM terpakai    : {mem.used / 1024**3:.2f} GB")
    print(f"RAM tersedia    : {mem.available / 1024**3:.2f} GB")
    print(f"RAM terpakai    : {mem.percent}%")
    print(f"CPU saat ini    : {psutil.cpu_percent(interval=2)}%")
except ImportError:
    print("psutil belum terpasang; memakai informasi Linux.")
    if Path("/proc/meminfo").exists():
        print(subprocess.run(
            ["bash", "-lc", "free -h"],
            capture_output=True, text=True
        ).stdout)

print("\n=== 2. RUANG PENYIMPANAN ===")
path = Path("/home/rzangef/SSD-PPE")
usage = shutil.disk_usage(path)
print(f"Total disk : {usage.total / 1024**3:.2f} GB")
print(f"Terpakai   : {usage.used / 1024**3:.2f} GB")
print(f"Tersedia   : {usage.free / 1024**3:.2f} GB")

print("\n=== 3. SUHU CPU (JIKA TERSEDIA) ===")
try:
    result = subprocess.run(
        ["bash", "-lc", "sensors"],
        capture_output=True, text=True, timeout=5
    )
    if result.returncode == 0 and result.stdout.strip():
        print(result.stdout[:4000])
    else:
        print("Data suhu tidak tersedia lewat perintah sensors.")
except Exception:
    print("Perintah sensors tidak tersedia.")

print("\n=== 4. STATUS WSL ===")
try:
    result = subprocess.run(
        ["bash", "-lc", "uptime"],
        capture_output=True, text=True, timeout=5
    )
    print(result.stdout.strip())
except Exception as e:
    print(e)

=== 1. SISTEM DAN CPU ===
Jumlah CPU logis: 4
RAM total       : 7.76 GB
RAM terpakai    : 1.15 GB
RAM tersedia    : 6.60 GB
RAM terpakai    : 14.9%
CPU saat ini    : 0.5%

=== 2. RUANG PENYIMPANAN ===
Total disk : 1006.85 GB
Terpakai   : 25.62 GB
Tersedia   : 930.02 GB

=== 3. SUHU CPU (JIKA TERSEDIA) ===
Data suhu tidak tersedia lewat perintah sensors.

=== 4. STATUS WSL ===
13:53:58 up  3:37,  1 user,  load average: 0.08, 0.02, 0.01


In [8]:
# Cek RAM WSL

import subprocess
from pathlib import Path

print("=== MEMORI WSL ===")
print(subprocess.run(
    ["bash", "-lc", "free -h && echo && cat /proc/meminfo | head -8"],
    capture_output=True, text=True
).stdout)

print("\n=== KONFIGURASI WSL ===")
result = subprocess.run(
    ["bash", "-lc",
     "cat /proc/mounts | grep -i drvfs | head -3; "
     "cat /proc/cpuinfo | grep -m1 'model name'"],
    capture_output=True, text=True
)
print(result.stdout)

print("\n=== SUHU CPU ===")
print("Jika sensors tidak tersedia, suhu dapat diperiksa melalui")
print("aplikasi pemantau perangkat Windows atau BIOS/UEFI.")

=== MEMORI WSL ===
               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       1.0Gi       6.3Gi       3.0Mi       509Mi       6.6Gi
Swap:          4.0Gi          0B       4.0Gi

MemTotal:        8133876 kB
MemFree:         6582196 kB
MemAvailable:    6929988 kB
Buffers:            4732 kB
Cached:           457832 kB
SwapCached:            0 kB
Active:            88880 kB
Inactive:        1144004 kB


=== KONFIGURASI WSL ===
C:\134 /mnt/c 9p rw,noatime,aname=drvfs;path=C:\;uid=1000;gid=1000;symlinkroot=/mnt/,cache=5,access=client,msize=65536,trans=fd,rfd=6,wfd=6 0 0
D:\134 /mnt/d 9p rw,noatime,aname=drvfs;path=D:\;uid=1000;gid=1000;symlinkroot=/mnt/,cache=5,access=client,msize=65536,trans=fd,rfd=6,wfd=6 0 0
E:\134 /mnt/e 9p rw,noatime,aname=drvfs;path=E:\;uid=1000;gid=1000;symlinkroot=/mnt/,cache=5,access=client,msize=65536,trans=fd,rfd=6,wfd=6 0 0
model name	: AMD Ryzen 7 7730U with Radeon Graphics


=== SUHU CPU ===
Jika sensors tidak t

In [1]:
# Cek checkpoint hasil training tersedia atau tidak
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")

MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
VAL_RECORD = PROJECT / "dataset/tfrecord_yolo_split/val.record"
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"

print("=== PEMERIKSAAN FILE ===")
for name, path in [
    ("Folder model", MODEL_DIR),
    ("Checkpoint state", MODEL_DIR / "checkpoint"),
    ("Konfigurasi training", CONFIG),
    ("Data validasi", VAL_RECORD),
    ("Label map", LABEL_MAP),
]:
    print(f"{name}: {'ADA' if path.exists() else 'TIDAK ADA'}")
    print(f"  {path}")

state_file = MODEL_DIR / "checkpoint"
if state_file.exists():
    print("\n=== ISI CHECKPOINT STATE ===")
    print(state_file.read_text())

print("\n=== CHECKPOINT INDEX ===")
for path in sorted(MODEL_DIR.glob("ckpt-*.index")):
    print(path.name)

=== PEMERIKSAAN FILE ===
Folder model: ADA
  /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split
Checkpoint state: ADA
  /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/checkpoint
Konfigurasi training: ADA
  /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
Data validasi: ADA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
Label map: ADA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt

=== ISI CHECKPOINT STATE ===
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1

In [2]:
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")

MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
VAL_RECORD = PROJECT / "dataset/tfrecord_yolo_split/val.record"
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"

print("PROJECT   :", PROJECT, PROJECT.exists())
print("CONFIG    :", CONFIG.exists(), CONFIG)
print("VAL_RECORD:", VAL_RECORD.exists(), VAL_RECORD)
print("LABEL_MAP :", LABEL_MAP.exists(), LABEL_MAP)
print("MODEL_DIR :", MODEL_DIR.exists(), MODEL_DIR)

PROJECT   : /home/rzangef/SSD-PPE True
CONFIG    : True /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
VAL_RECORD: True /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
LABEL_MAP : True /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt
MODEL_DIR : True /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split


In [3]:
from pathlib import Path
import re

# Lokasi proyek SSD
PROJECT = Path("/home/rzangef/SSD-PPE")

# Lokasi file konfigurasi dan dataset
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
VAL_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
VAL_RECORD = PROJECT / "dataset/tfrecord_yolo_split/val.record"
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"

print("=== PEMERIKSAAN FILE ===")
for name, path in [
    ("PROJECT", PROJECT),
    ("CONFIG training", CONFIG),
    ("VAL_CONFIG", VAL_CONFIG),
    ("VAL_RECORD", VAL_RECORD),
    ("LABEL_MAP", LABEL_MAP),
]:
    print(f"{name}: {'ADA' if path.exists() else 'TIDAK ADA'}")
    print(f"  {path}")

=== PEMERIKSAAN FILE ===
PROJECT: ADA
  /home/rzangef/SSD-PPE
CONFIG training: ADA
  /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
VAL_CONFIG: TIDAK ADA
  /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
VAL_RECORD: ADA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
LABEL_MAP: ADA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt


In [4]:

from pathlib import Path
import re

PROJECT = Path("/home/rzangef/SSD-PPE")

TRAIN_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
VAL_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"
MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"

def show_config_checks(path):
    print(f"\n=== {path.name} ===")
    if not path.exists():
        print("FILE TIDAK DITEMUKAN")
        return

    lines = path.read_text(errors="replace").splitlines()
    patterns = re.compile(
        r"num_classes|num_steps|batch_size|"
        r"fine_tune_checkpoint|fine_tune_checkpoint_type|"
        r"input_path|label_map_path|"
        r"image_resizer|height:|width:|"
        r"num_epochs|num_examples|metrics_set|"
        r"use_moving_averages|shuffle:"
    )

    for i, line in enumerate(lines):
        if patterns.search(line):
            print(f"{i+1}: {line.strip()}")

show_config_checks(TRAIN_CONFIG)
show_config_checks(VAL_CONFIG)

print("\n=== LABEL MAP ===")
if LABEL_MAP.exists():
    print(LABEL_MAP.read_text())
else:
    print("Label map tidak ditemukan.")

print("\n=== CHECKPOINT STATE ===")
state_file = MODEL_DIR / "checkpoint"
if state_file.exists():
    print(state_file.read_text())
else:
    print("File checkpoint tidak ditemukan.")

print("\n=== FILE CHECKPOINT ===")
if MODEL_DIR.exists():
    for p in sorted(MODEL_DIR.glob("ckpt-*")):
        print(p.name, round(p.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("Folder model tidak ditemukan.")



=== pipeline_ppe_yolo_split_stage5.config ===
3: num_classes: 8
4: image_resizer {
6: height: 320
7: width: 320
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps: 5945
171: fine_tune_checkpoint_type: "detection"
172: fine_tune_checkpoint_version: V2
175: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
177: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
181: metrics_set: "coco_detection_metrics"
182: use_moving_averages: false
185: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
186: shuffle: false
187: num_epochs: 1
189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"

=== pipeline_ppe_yolo_split_val.config ===
3: num_classes: 8
4: image_resizer {
6: height: 320
7: width: 320
135: batch_size: 1
165: fine_tune_checkpoint: 

In [5]:

from pathlib import Path
from collections import Counter
import tensorflow as tf

PROJECT = Path("/home/rzangef/SSD-PPE")
DATA_DIR = PROJECT / "dataset/tfrecord_yolo_split"

records = {
    "TRAIN": DATA_DIR / "train.record",
    "VALIDASI": DATA_DIR / "val.record",
}

label_names = {
    1: "person",
    2: "vest",
    3: "yellow",
    4: "white",
    5: "glass",
    6: "blue",
    7: "red",
    8: "head",
}

feature_description = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
}

for split, record_path in records.items():
    print(f"\n=== {split} ===")
    print("File tersedia:", record_path.exists())

    if not record_path.exists():
        continue

    class_counts = Counter()
    total_images = 0
    empty_images = 0
    invalid_ids = Counter()
    bbox_mismatches = 0

    for raw_record in tf.data.TFRecordDataset(str(record_path)):
        example = tf.io.parse_single_example(
            raw_record, feature_description
        )
        total_images += 1

        ids = tf.sparse.to_dense(
            example["image/object/class/label"]
        ).numpy().tolist()

        bbox_lengths = [
            tf.size(tf.sparse.to_dense(example[key])).numpy()
            for key in [
                "image/object/bbox/xmin",
                "image/object/bbox/xmax",
                "image/object/bbox/ymin",
                "image/object/bbox/ymax",
            ]
        ]

        if len(set(bbox_lengths + [len(ids)])) != 1:
            bbox_mismatches += 1

        if not ids:
            empty_images += 1

        for class_id in ids:
            class_counts[class_id] += 1
            if class_id not in label_names:
                invalid_ids[class_id] += 1

    print("Jumlah gambar:", total_images)
    print("Gambar tanpa objek:", empty_images)
    print("Jumlah objek:", sum(class_counts.values()))
    print("ID kelas di luar rentang 1–8:", dict(invalid_ids))
    print("Gambar dengan jumlah label dan bbox tidak cocok:", bbox_mismatches)
    print("Jumlah objek per kelas:")

    for class_id, name in label_names.items():
        print(f"  {class_id} - {name}: {class_counts[class_id]}")


2026-10-21 08:56:28.901580: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:56:28.936314: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-21 08:56:28.936359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-21 08:56:28.937637: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-21 08:56:28.944084: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:56:28.944822: I tensorflow/core/platform/cpu_feature_guard.cc:1


=== TRAIN ===
File tersedia: True
Jumlah gambar: 1189
Gambar tanpa objek: 1
Jumlah objek: 8008
ID kelas di luar rentang 1–8: {}
Gambar dengan jumlah label dan bbox tidak cocok: 0
Jumlah objek per kelas:
  1 - person: 3220
  2 - vest: 1474
  3 - yellow: 841
  4 - white: 899
  5 - glass: 380
  6 - blue: 343
  7 - red: 364
  8 - head: 487

=== VALIDASI ===
File tersedia: True
Jumlah gambar: 170
Gambar tanpa objek: 0
Jumlah objek: 1211
ID kelas di luar rentang 1–8: {}
Gambar dengan jumlah label dan bbox tidak cocok: 0
Jumlah objek per kelas:
  1 - person: 490
  2 - vest: 233
  3 - yellow: 112
  4 - white: 130
  5 - glass: 49
  6 - blue: 55
  7 - red: 50
  8 - head: 92


In [6]:
from pathlib import Path

root = Path("/home/rzangef/SSD-PPE")

for folder in [root / "logs", root / "evaluation"]:
    print(f"\n=== {folder} ===")
    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file() and (
                "eval" in p.name.lower()
                or "train" in p.name.lower()
                or p.name.endswith(".log")
            ):
                print(p)


=== /home/rzangef/SSD-PPE/logs ===
/home/rzangef/SSD-PPE/logs/ssd_stage5_validation.log
/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log

=== /home/rzangef/SSD-PPE/evaluation ===


In [7]:
from pathlib import Path

log_path = Path("/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log")

if not log_path.exists():
    print("Log training tidak ditemukan.")
else:
    lines = log_path.read_text(errors="replace").splitlines()

    print("Jumlah baris log:", len(lines))
    print("\n=== Baris terkait checkpoint dan metrik ===")

    keywords = [
        "checkpoint",
        "global step",
        "loss",
        "precision",
        "recall",
        "DetectionBoxes",
        "mAP",
        "evaluation",
        "error",
    ]

    matches = [
        (i + 1, line)
        for i, line in enumerate(lines)
        if any(word.lower() in line.lower() for word in keywords)
    ]

    for line_number, line in matches[-80:]:
        print(f"{line_number}: {line}")

Jumlah baris log: 784

=== Baris terkait checkpoint dan metrik ===
667: INFO:tensorflow:{'Loss/classification_loss': 0.5513638,
668:  'Loss/localization_loss': 0.41088167,
669:  'Loss/regularization_loss': 0.15335605,
670:  'Loss/total_loss': 1.1156015,
672: I1020 15:26:29.003806 140488109375488 model_lib_v2.py:708] {'Loss/classification_loss': 0.5513638,
673:  'Loss/localization_loss': 0.41088167,
674:  'Loss/regularization_loss': 0.15335605,
675:  'Loss/total_loss': 1.1156015,
679: INFO:tensorflow:{'Loss/classification_loss': 0.49897423,
680:  'Loss/localization_loss': 0.3794616,
681:  'Loss/regularization_loss': 0.15335588,
682:  'Loss/total_loss': 1.0317917,
684: I1020 15:26:59.018666 140488109375488 model_lib_v2.py:708] {'Loss/classification_loss': 0.49897423,
685:  'Loss/localization_loss': 0.3794616,
686:  'Loss/regularization_loss': 0.15335588,
687:  'Loss/total_loss': 1.0317917,
691: INFO:tensorflow:{'Loss/classification_loss': 0.7006608,
692:  'Loss/localization_loss': 0.7139

In [8]:
from pathlib import Path

root = Path("/home/rzangef/SSD-PPE")

paths = [
    root / "models/pipeline_ppe_yolo_split_stage5.config",
    root / "models/pipeline_ppe_yolo_split_val.config",
    root / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/checkpoint",
    root / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/checkpoint",
    root / "evaluation/ssd_yolo_split_val/eval",
]

for path in paths:
    print(f"\n{path}")
    print("Tersedia:", path.exists())

    if path.is_file() and path.name == "checkpoint":
        print(path.read_text(errors="replace"))

    elif path.is_dir():
        for item in sorted(path.iterdir()):
            print(" -", item.name)


/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
Tersedia: True

/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
Tersedia: True

/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/checkpoint
Tersedia: True
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1792484159.868539
all_model_checkpoint_timestamps: 1792484458.692329
all_model_checkpoint_timestamps: 1792484758.62851
all_model_checkpoint_timestamps: 1792485066.0702882
last_preserved_timestamp: 1792483214.359389


/home/rzangef/SSD-PPE/training/ssd_mobi

In [9]:
import sys
import tensorflow as tf
import keras

print("Python executable :", sys.executable)
print("TensorFlow version:", tf.__version__)
print("TensorFlow path   :", tf.__file__)
print("Keras version     :", keras.__version__)
print("Keras path        :", keras.__file__)

Python executable : /home/rzangef/SSD-PPE/.venv/bin/python
TensorFlow version: 2.15.1
TensorFlow path   : /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/__init__.py
Keras version     : 2.15.0
Keras path        : /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/keras/__init__.py


In [10]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

print("\n=== Pemeriksaan API ===")
print(
    "tf.keras.layers.experimental tersedia:",
    hasattr(tf.keras.layers, "experimental")
)
print(
    "SyncBatchNormalization tersedia:",
    hasattr(tf.keras.layers, "SyncBatchNormalization")
)
print(
    "tf.keras berasal dari:",
    tf.keras.__file__ if hasattr(tf.keras, "__file__") else "tidak tersedia"
)

TensorFlow: 2.15.1
Keras: 2.15.0

=== Pemeriksaan API ===
tf.keras.layers.experimental tersedia: True
SyncBatchNormalization tersedia: False
tf.keras berasal dari: /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/keras/api/_v2/keras/__init__.py


In [11]:
import tensorflow as tf

# Periksa apakah implementasi alternatif tersedia
print(
    "tf.keras.layers.SyncBatchNormalization:",
    hasattr(tf.keras.layers, "SyncBatchNormalization")
)

print(
    "tf.keras.layers.BatchNormalization:",
    hasattr(tf.keras.layers, "BatchNormalization")
)

print(
    "tf.keras.layers.experimental:",
    hasattr(tf.keras.layers, "experimental")
)

tf.keras.layers.SyncBatchNormalization: False
tf.keras.layers.BatchNormalization: True
tf.keras.layers.experimental: True


In [12]:
import tensorflow as tf

try:
    from tensorflow.python.keras.layers import SyncBatchNormalization
    print("Implementasi internal tersedia:", SyncBatchNormalization)
except Exception as e:
    print("Implementasi internal tidak tersedia:", repr(e))

Implementasi internal tidak tersedia: ImportError("cannot import name 'SyncBatchNormalization' from 'tensorflow.python.keras.layers' (/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/keras/layers/__init__.py)")


In [1]:
# Jumat, 23 Oktober 2026
# Melakukan pengujian tahap selanjutnya

In [3]:
# Cek kondisi terakhir eksperimen
from pathlib import Path
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")

def run_command(command):
    result = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

print("=== CHECKPOINT TRAINING ===")
run_command(
    "cat training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/checkpoint"
)

print("\n=== PROSES TRAINING / EVALUASI ===")
run_command(
    "ps -ef | grep -E 'model_main_tf2.py|eval' | grep -v grep"
)

print("\n=== FILE DATASET ===")
run_command("ls -lh dataset/tfrecord_yolo_split/")

print("\n=== LOG EVALUASI VALIDASI ===")
run_command("tail -n 30 logs/ssd_stage5_validation.log")

=== CHECKPOINT TRAINING ===
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1792484159.868539
all_model_checkpoint_timestamps: 1792484458.692329
all_model_checkpoint_timestamps: 1792484758.62851
all_model_checkpoint_timestamps: 1792485066.0702882
last_preserved_timestamp: 1792483214.359389


=== PROSES TRAINING / EVALUASI ===


=== FILE DATASET ===
total 100M
-rw-r--r-- 1 rzangef rzangef  301 Oct  8 20:14 label_map.pbtxt
-rw-r--r-- 1 rzangef rzangef  20M Oct 20 11:42 test.record
-rw-r--r-- 1 rzangef rzangef  70M Oct 20 11:41 train.record
-rw-r--r-- 1 rzangef rzan

In [4]:
# Periksa konfigurasi validasi dan test
from pathlib import Path
import re

PROJECT = Path("/home/rzangef/SSD-PPE")

files = [
    PROJECT / "models/pipeline_ppe_yolo_split_stage5.config",
    PROJECT / "models/pipeline_ppe_yolo_split_val.config",
    PROJECT / "models/pipeline_ppe_yolo_split_test.config",
]

patterns = [
    "num_classes",
    "batch_size",
    "num_steps",
    "fine_tune_checkpoint",
    "train_input_reader",
    "eval_input_reader",
    "input_path",
    "label_map_path",
    "num_examples",
]

for path in files:
    print("\n" + "=" * 65)
    print("FILE:", path.name)
    print("ADA:", path.exists())

    if not path.exists():
        continue

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(p in line for p in patterns):
            print(f"{i+1}: {line.strip()}")

print("\n" + "=" * 65)
print("FILE DATASET TFRECORD:")
dataset_dir = PROJECT / "dataset/tfrecord_yolo_split"

for path in sorted(dataset_dir.glob("*")):
    print(repr(path.name), "-", path.stat().st_size, "bytes")


FILE: pipeline_ppe_yolo_split_stage5.config
ADA: True
3: num_classes: 8
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps: 5945
171: fine_tune_checkpoint_type: "detection"
172: fine_tune_checkpoint_version: V2
174: train_input_reader {
175: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
177: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
184: eval_input_reader {
185: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"

FILE: pipeline_ppe_yolo_split_val.config
ADA: True
3: num_classes: 8
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps:

In [5]:
# MEMPERSIAPKAN UNTUK MELAKUKAN evaluasi pada Data Test

In [6]:
# Pastikan konfigurasi test siap
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")

configs = [
    PROJECT / "models/pipeline_ppe_yolo_split_val.config",
    PROJECT / "models/pipeline_ppe_yolo_split_test.config",
]

for path in configs:
    print("\n" + "=" * 60)
    print("FILE:", path.name)

    lines = path.read_text().splitlines()

    # Tampilkan bagian konfigurasi evaluasi dan pembaca data
    for i, line in enumerate(lines):
        if any(key in line for key in [
            "eval_config {",
            "num_examples:",
            "metrics_set:",
            "eval_input_reader {",
            "input_path:",
            "label_map_path:",
        ]):
            start = max(0, i - 2)
            end = min(len(lines), i + 3)
            print(f"\nBaris {i+1}:")
            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")

print("\n" + "=" * 60)
print("CHECKPOINT YANG TERSEDIA:")

checkpoint_dir = (
    PROJECT
    / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

for path in sorted(checkpoint_dir.glob("ckpt-*")):
    print(path.name)


FILE: pipeline_ppe_yolo_split_val.config

Baris 175:
173: }
174: train_input_reader {
175:   label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
176:   tf_record_input_reader {
177:     input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"

Baris 177:
175:   label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
176:   tf_record_input_reader {
177:     input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
178:   }
179: }

Baris 180:
178:   }
179: }
180: eval_config {
181:   metrics_set: "coco_detection_metrics"
182:   use_moving_averages: false

Baris 181:
179: }
180: eval_config {
181:   metrics_set: "coco_detection_metrics"
182:   use_moving_averages: false
183: 

Baris 184:
182:   use_moving_averages: false
183: 
184:   num_examples: 170
185: }
186: eval_input_reader {

Baris 186:
184:   num_examples: 170
185: }
186: eval_input_reader {
187:   label_map_path: "/home/rzangef/SS

In [7]:
# Pastikan lokasi output training
from pathlib import Path

p = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

print("Direktori ditemukan:", p.is_dir())
print("Checkpoint index ckpt-7:", (p / "ckpt-7.index").exists())
print("File checkpoint state:", (p / "checkpoint").exists())

Direktori ditemukan: True
Checkpoint index ckpt-7: True
File checkpoint state: True


In [8]:
# Periksa jumlah gambar test
from pathlib import Path

config = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

lines = config.read_text().splitlines()

for i, line in enumerate(lines):
    if "num_examples:" in line or "test.record" in line:
        print(f"Baris {i+1}: {line.strip()}")

Baris 189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"


In [9]:
# Belum diketahui num_example : 340

In [10]:
# Periksa bagian eval_config
from pathlib import Path

path = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

lines = path.read_text().splitlines()

for i, line in enumerate(lines):
    if "eval_config {" in line:
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(i, min(i+15, len(lines)))
        ))
        break

180: eval_config {
181:   metrics_set: "coco_detection_metrics"
182:   use_moving_averages: false
183: }
184: eval_input_reader {
185:   label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
186:   shuffle: false
187:   num_epochs: 1
188:   tf_record_input_reader {
189:     input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"
190:   }
191: }


In [11]:
# Menambahkan num_example : 340 pada konfigurasi data test
from pathlib import Path

path = Path(
    "/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config"
)

config = path.read_text()

old = '''eval_config {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
}'''

new = '''eval_config {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
  num_examples: 340
}'''

if new in config:
    print("num_examples: 340 sudah tersedia. Tidak ada perubahan.")
elif old in config:
    backup = path.with_suffix(".config.bak")
    backup.write_text(config)
    path.write_text(config.replace(old, new, 1))
    print("Konfigurasi berhasil diperbarui.")
    print("Cadangan:", backup)
else:
    print("Blok eval_config tidak cocok. File tidak diubah.")

# Periksa hasilnya
lines = path.read_text().splitlines()
for i, line in enumerate(lines):
    if "eval_config {" in line:
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(i, min(i + 8, len(lines)))
        ))
        break

Konfigurasi berhasil diperbarui.
Cadangan: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config.bak
180: eval_config {
181:   metrics_set: "coco_detection_metrics"
182:   use_moving_averages: false
183:   num_examples: 340
184: }
185: eval_input_reader {
186:   label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
187:   shuffle: false
